# Exercise 2 — format_docs and build_retrieval_prompt

`format_docs` turns a list of Documents into a numbered context block for an LLM prompt.  `build_retrieval_prompt` wraps that context into a full RAG prompt — a fixed-pipeline approach (Day 13 pattern) that the agentic version will extend.

In [ ]:
import json
from dataclasses import dataclass, field
@dataclass
class Document:
    content: str
    metadata: dict = field(default_factory=dict)

class SimpleRetriever:
    def __init__(self):
        self._docs = []
    def add(self, doc):
        self._docs.append(doc); return self
    def add_all(self, docs):
        for d in docs: self._docs.append(d)
        return self
    def _score(self, query, doc):
        q = set(query.lower().split())
        d = set(doc.content.lower().split())
        return len(q & d) / (len(q | d) + 1e-9)
    def search(self, query, top_k=3):
        if not self._docs: return []
        return sorted(self._docs, key=lambda doc: self._score(query, doc), reverse=True)[:top_k]
    def __len__(self): return len(self._docs)

# ── Exercise: implement format_docs and build_retrieval_prompt ───────────────

def format_docs(docs):
    # TODO: if docs is empty, return "No documents found."
    # For each doc (1-indexed), get source from doc.metadata.get("source", f"doc{i}")
    # Append "[i] (source) content" to lines
    # Return lines joined with "\n"
    return "No documents found."


def build_retrieval_prompt(question, docs):
    # TODO: call format_docs(docs) to get context
    # Build a system message instructing the model to answer from the documents
    # and cite doc numbers like [1]
    # Return [{"role": "system", ...}, {"role": "user", "content": "Documents:\n...\n\nQuestion: ..."}]
    return [{"role": "system", "content": ""}, {"role": "user", "content": str(question)}]


### Checks

In [ ]:
checks = 0

docs = [
    Document("Python is a programming language.", {"source": "wiki"}),
    Document("Python uses indentation.", {"source": "docs"}),
]

# 1 — format_docs with empty list
try:
    result = format_docs([])
    assert result == "No documents found."
    checks += 1; print("✅ 1 format_docs([]) returns 'No documents found.'")
except Exception as e:
    print("❌ 1:", e)

# 2 — format_docs includes doc numbers
try:
    text = format_docs(docs)
    assert "[1]" in text and "[2]" in text
    checks += 1; print("✅ 2 format_docs includes [1], [2] numbering")
except Exception as e:
    print("❌ 2:", e)

# 3 — format_docs includes source from metadata
try:
    text = format_docs(docs)
    assert "wiki" in text
    checks += 1; print("✅ 3 format_docs includes source from metadata")
except Exception as e:
    print("❌ 3:", e)

# 4 — build_retrieval_prompt returns list of 2 messages
try:
    prompt = build_retrieval_prompt("What is Python?", docs)
    assert isinstance(prompt, list) and len(prompt) == 2
    assert prompt[0]["role"] == "system"
    assert prompt[1]["role"] == "user"
    checks += 1; print("✅ 4 build_retrieval_prompt returns [system, user] messages")
except Exception as e:
    print("❌ 4:", e)

# 5 — build_retrieval_prompt includes question and doc content
try:
    prompt = build_retrieval_prompt("What is Python?", docs)
    combined = " ".join(m["content"] for m in prompt)
    assert "Python" in combined and "What is Python?" in combined
    checks += 1; print("✅ 5 prompt includes question and document content")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
